In [ ]:
import os
import re
import glob
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

CARPETA_ENTRADA = "entrada"
CARPETA_BD = "bd"
CARPETA_SALIDA = "salida"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

def encontrar_extracto_mas_reciente(carpeta=CARPETA_ENTRADA):
    candidatos = glob.glob(os.path.join(carpeta, "cl??????.xlsx"))
    con_fecha = []
    for ruta in candidatos:
        m = re.search(r"cl(\d{2})(\d{2})(\d{2})\.xlsx$", os.path.basename(ruta))
        if m:
            dd, mm, yy = m.groups()
            try:
                con_fecha.append((datetime.strptime(f"{dd}{mm}{yy}", "%d%m%y"), ruta))
            except ValueError:
                continue
    if not con_fecha:
        raise FileNotFoundError(f"No se encontró ningún cl{{DDMMYY}}.xlsx en {carpeta}")
    con_fecha.sort()
    return con_fecha[-1]

fecha_extracto, RUTA_EXTRACTO_SP = encontrar_extracto_mas_reciente()
RUTA_BD_VOLUMEN = os.path.join(CARPETA_BD, "campuslab_volumen.db")

print(f"SharePoint: {RUTA_EXTRACTO_SP} (del {fecha_extracto:%d/%m/%Y})")
print(f"Volumen:    {RUTA_BD_VOLUMEN} ({'existe' if os.path.exists(RUTA_BD_VOLUMEN) else 'NO ENCONTRADO'})")

In [ ]:
RAIZ_SITIO = "sites/ProyectoBLP/Documentos compartidos"

df_sp = pd.read_excel(RUTA_EXTRACTO_SP)
df_sp["ruta_rel"] = df_sp["Ruta de acceso"].str.replace(RAIZ_SITIO, "", regex=False).str.strip("/")
df_sp["ruta_completa"] = df_sp["ruta_rel"] + "/" + df_sp["Nombre"]

fotos_sp = df_sp[
    (df_sp["Tipo de elemento"] == "Elemento") &
    (df_sp["ruta_rel"].str.startswith("Bases de datos/FOTOS"))
].copy()
print(f"Archivos en SharePoint > Bases de datos/FOTOS: {len(fotos_sp):,}")

con = sqlite3.connect(RUTA_BD_VOLUMEN)
vol = pd.read_sql("SELECT nombre_archivo, tamano_bytes, ruta_completa AS ruta_volumen FROM staging_escaneo", con)
con.close()
print(f"Archivos escaneados en el volumen: {len(vol):,}")

In [ ]:
vol_set_exacto = set(zip(vol["nombre_archivo"], vol["tamano_bytes"]))
vol_nombres = set(vol["nombre_archivo"])

def clasificar(row):
    clave = (row["Nombre"], row["Tamaño de archivo"])
    if clave in vol_set_exacto:
        return "Ya en el volumen"
    elif row["Nombre"] in vol_nombres:
        return "Revisar: nombre coincide, tamaño distinto"
    else:
        return "Candidato a migrar"

fotos_sp["estado_cruce"] = fotos_sp.apply(clasificar, axis=1)

resumen = fotos_sp["estado_cruce"].value_counts()
print("=== Resultado del cruce ===")
for estado, n in resumen.items():
    print(f"  {n:5,d}  {estado}")

In [ ]:
def fmt_size(b):
    if pd.isna(b):
        return ""
    b = float(b)
    for unit in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f} {unit}"
        b /= 1024
    return f"{b:.1f} TB"

fotos_sp["Tamaño legible"] = fotos_sp["Tamaño de archivo"].apply(fmt_size)

pendientes = fotos_sp[fotos_sp["estado_cruce"] != "Ya en el volumen"][
    ["estado_cruce", "Nombre", "Tamaño legible", "Modificado", "Modificado por", "ruta_completa"]
].sort_values(["estado_cruce", "Nombre"]).reset_index(drop=True)

HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF")
HEADER_FILL = PatternFill(start_color="2F5496", end_color="2F5496", fill_type="solid")
BODY_FONT = Font(name="Arial", size=10)
REVISAR_FILL = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")

wb = Workbook()
ws1 = wb.active
ws1.title = "Resumen"
ws1["A1"] = "Cruce SharePoint (Bases de datos/FOTOS) vs. Volumen"
ws1["A1"].font = Font(name="Arial", bold=True, size=12)
r = 3
for estado, n in resumen.items():
    ws1.cell(row=r, column=1, value=f"{estado}: {n:,}").font = Font(name="Arial", size=11, bold=True)
    r += 1
ws1[f"A{r+1}"] = "El cruce es por nombre + tamaño exacto. No detecta copias renombradas ni ediciones."
ws1[f"A{r+1}"].font = Font(name="Arial", italic=True, size=10, color="808080")
ws1.column_dimensions["A"].width = 90

ws2 = wb.create_sheet("Por migrar o revisar")
cols = list(pendientes.columns)
for c, h in enumerate(cols, start=1):
    cell = ws2.cell(row=1, column=c, value=h); cell.font, cell.fill = HEADER_FONT, HEADER_FILL
for r, row in enumerate(pendientes.itertuples(index=False), start=2):
    for c, val in enumerate(row, start=1):
        ws2.cell(row=r, column=c, value=str(val) if pd.notna(val) else "").font = BODY_FONT
    if "Revisar" in str(row[0]):
        for c in range(1, len(cols) + 1):
            ws2.cell(row=r, column=c).fill = REVISAR_FILL
ws2.freeze_panes, ws2.auto_filter.ref = "A2", ws2.dimensions
for i, w in enumerate([32, 40, 12, 18, 26, 90], start=1):
    ws2.column_dimensions[get_column_letter(i)].width = w

ruta_out = os.path.join(CARPETA_SALIDA, "Cruce_SharePoint_vs_Volumen.xlsx")
wb.save(ruta_out)
print(f"Guardado: {ruta_out}")
print(f"Filas en 'Por migrar o revisar': {len(pendientes):,}")